## Introduccion

Este proyecto tiene como objetivo demostrar las habilidades adquiridas durante la clase **Proyecto de Ciencia de Datos**, impartida por el profesor [Cristian Camilo Zapata Zuluaga](https://github.com/zapatacc/), en el quinto semestre de la carrera de Ingeniería en Ciencia de Datos en el Instituto Tecnológico y de Estudios Superiores de Occidente (ITESO). A través de este trabajo, se busca analizar un conjunto de datos de telemetría ambiental recopilados mediante dispositivos IoT, comprender la estructura de los proyectos de ciencia de datos, implementar modelos de *machine learning* para predecir la temperatura en diversas condiciones ambientales, y, además, dominar la orquestación de flujos mediante *pipelines* con Prefect para automatizar los procesos de entrenamiento y registro de modelos.

El enfoque principal del proyecto es construir un sistema robusto para el manejo y análisis de datos ambientales, integrando herramientas como **MLflow** y **Dagshub** para el seguimiento y gestión de experimentos, y empleando técnicas avanzadas de optimización de modelos para obtener predicciones precisas. Finalmente, este trabajo resalta la importancia de llevar los resultados a un entorno interactivo mediante el desarrollo de una aplicación con **Streamlit** y **FastAPI**.

---

### Índice

1. **Entendimiento del problema**
2. **Importancia del Problema**

2. **Análisis Exploratorio de los Datos (EDA)**
3. **Flujos de Datos y Estructura del Proyecto**
   - Organización de archivos y carpetas
   - Documentación técnica
4. **Entrenamiento de Modelos**
   - Experimentación y *hyperparameter tuning*
   - Comparación y selección de modelos (*Champion* y *Challenger*)
   - Registro en el Model Registry
5. **Pipelines de Automatización y Tracking**
   - Orquestación con Prefect
   - Ejecución local y configuración con MLflow
6. **Desarrollo de la Aplicación**
   - **Interfaz de Usuario (Frontend)**:
     - Diseño con Streamlit
   - **Modelo y API (Backend)**:
     - Implementación con FastAPI 
7. **Conclusiones y Reflexiones Finales**

---

### 1. Entendimiento del Problema

El proyecto se centra en el análisis de datos provenientes de dispositivos IoT que monitorean condiciones ambientales específicas en diversas ubicaciones durante el período del **07/12/2020 al 07/19/2020**. Estos dispositivos registran mediciones clave como temperatura, humedad, gases, y actividad detectada mediante sensores, los cuales se transmiten utilizando el protocolo **MQTT**, un estándar eficiente para la comunicación en redes de sensores.  

Cada dispositivo presenta características únicas en cuanto a las condiciones ambientales que monitorea:
- **`00:0f:00:70:91:0a`:** Un entorno frío y húmedo con condiciones estables.
- **`1c:bf:ce:15:ec:4d`:** Un entorno con alta variabilidad en temperatura y humedad.
- **`b8:27:eb:bf:9d:51`:** Un entorno cálido y seco, también con condiciones estables.

El objetivo principal es **predecir la temperatura** en función de las demás variables medidas (como humedad, niveles de gases y otros factores), utilizando técnicas de *machine learning*. Este problema se enmarca en el ámbito de la **regresión supervisada**, ya que la temperatura es una variable continua y los datos están etiquetados.

---

### 2. Importancia del Problema

1. **Impacto en la Monitorización Ambiental:**  
   La predicción precisa de la temperatura basada en datos de sensores IoT tiene aplicaciones cruciales en diversos sectores, como:
   - **Agricultura:** Ayuda a monitorear y controlar las condiciones climáticas para cultivos sensibles a la temperatura.
   - **Salud:** Detectar condiciones que pueden exacerbar problemas de salud relacionados con el clima, como la exposición a niveles extremos de calor o frío.
   - **Energía:** Optimizar sistemas de calefacción, ventilación y aire acondicionado (HVAC) basados en predicciones precisas de temperatura.

2. **Automatización y Escalabilidad:**  
   Implementar un modelo de *machine learning* automatizado permite a las empresas o instituciones trabajar con grandes volúmenes de datos en tiempo real, proporcionando información valiosa para la toma de decisiones sin intervención manual.

3. **Desafíos Técnicos:**  
   Este problema no solo implica predecir una variable continua (temperatura), sino también lidiar con retos como:
   - **Datos variables:** Los dispositivos registran datos en entornos con diferentes condiciones, lo que añade complejidad al modelo.
   - **Limpieza de datos:** La detección y corrección de valores nulos o anómalos es esencial para evitar resultados sesgados.
   - **Optimización de hiperparámetros:** Garantizar que los modelos sean eficientes y precisos requiere ajustar configuraciones específicas.

---

### 4. Flujos de Datos y Estructura del Proyecto

__Organización de Archivos y Carpetas__

El proyecto está organizado en un sistema de directorios diseñado para maximizar la modularidad y escalabilidad, asegurando una separación clara de las diferentes etapas del flujo de trabajo. La estructura de carpetas es la siguiente:

```bash
PROYECTO-FINAL-CIENCIA-DATOS/
├── .venv/                      # Entorno virtual para gestionar dependencias
├── data/                       # Datos utilizados en el proyecto
│   ├── raw/                    # Datos crudos sin procesar
│   │   └── data.csv            # Archivo de datos originales
├── experiments/                # Scripts de automatización
│   └── pipeline.py             # Pipeline principal para entrenamiento y registro
├── INFORME_ESCRITO/            # Informes de análisis y resultados
│   ├── informe_escrito.ipynb   # Resumen inicial del análisis
│   └── informeFinal.ipynb      # Informe final con todos los resultados del proyecto
├── notebooks/                  # Notebooks de experimentación
│   ├── EDA.ipynb               # Análisis Exploratorio de Datos
│   ├── newModel.ipynb          # Entrenamiento y loggeo manual de modelos
│   └── updateExperiments.ipynb # Loggeo y seguimiento de experimentos en Dagshub
├── src/                        # Código fuente del proyecto
│   ├── model/                  # Implementación del backend para el modelo
│   │   ├── __init__.py
│   │   ├── api.py              # API desarrollada con FastAPI
│   │   ├── Dockerfile          # Configuración de contenedor para el backend
│   │   └── requirements.txt    # Dependencias necesarias para el backend
│   ├── data/                   # Scripts para procesamiento de datos
│   ├── UI/                     # Frontend desarrollado con Streamlit
├── docker-compose.yml          # Configuración para ejecutar servicios con Docker
├── .gitignore                  # Archivos ignorados por Git
├── LICENSE                     # Licencia del proyecto
├── pyproject.toml              # Configuración del entorno de desarrollo en Python
├── README.md                   # Documentación del proyecto
└── uv.lock                     # Archivo de bloqueo para dependencias
```

#### Descripción de Carpetas Clave

1. **`data/`:**  
   Contiene los datos en tres etapas:
   - **`raw/`:** Datos crudos obtenidos directamente del sistema de IoT.
   - Procesamiento adicional se realiza dentro de los scripts en `src/data`.

2. **`experiments/`:**  
   Contiene el archivo `pipeline.py`, que define el flujo automatizado del proyecto:
   - Lectura y limpieza de datos.
   - Entrenamiento y optimización de modelos.
   - Registro en **MLflow**.

3. **`notebooks/`:**  
   Diseñados para análisis exploratorio y experimentación independiente.
   - **EDA.ipynb:** Análisis inicial de los datos para detectar patrones.
   - **newModel.ipynb:** Entrenamiento manual de modelos y registro local.
   - **updateExperiments.ipynb:** Loggeo y gestión de modelos en Dagshub.

4. **`src/`:**  
   Incluye todo el código relacionado con la aplicación, dividido en:
   - **`model/`:** Backend del proyecto, desarrollado con FastAPI.
   - **`UI/`:** Interfaz de usuario creada con Streamlit para visualización.

#### Documentación Técnica

La documentación del proyecto está organizada en los siguientes archivos clave:

- **`README.md`:**  
  Contiene una descripción general del proyecto, incluyendo el propósito, estructura, instrucciones de instalación y detalles técnicos relevantes.

- **`informe_escrito.ipynb`:**  
  Primer informe que incluye el análisis exploratorio de los datos (EDA) y el diseño del pipeline inicial, con explicaciones detalladas y visualizaciones.

- **`requirements.txt`:**  
  Lista de dependencias necesarias para ejecutar el backend del proyecto, como FastAPI, MLflow y Prefect, entre otras.

- **`docker-compose.yml`:**  
  Archivo de configuración técnica para ejecutar tanto el backend como el frontend utilizando contenedores Docker, facilitando la implementación del proyecto en cualquier entorno.

Este enfoque asegura que todos los aspectos técnicos del proyecto estén documentados, permitiendo un fácil acceso a la información y facilitando la colaboración y el mantenimiento.

---

### 5. Entrenamiento de Modelos

#### **Experimentación y *Hyperparameter Tuning***
En esta etapa, se desarrollaron varios modelos de regresión para predecir la temperatura a partir de las demás variables del dataset. Los modelos evaluados incluyeron **Random Forest**, **LightGBM** y **CatBoost**. Cada modelo fue ajustado utilizando optimización de hiperparámetros a través de `hyperopt` y técnicas como validación cruzada para maximizar su desempeño.

Los pasos generales incluyen:
1. **Entrenamiento Inicial de Modelos:**
   - Entrenamiento de modelos base con hiperparámetros por defecto.
   - Registro de métricas como el **Root Mean Squared Error (RMSE)** en **MLflow**.
   - Registro del modelo entrenado como artefacto en **MLflow**.

2. **Optimización de Hiperparámetros:**
   - Se definieron espacios de búsqueda para cada modelo, ajustando parámetros clave como el número de estimadores, profundidad máxima, tasa de aprendizaje y otros.
   - Utilización de algoritmos de búsqueda como `Tree-structured Parzen Estimator (TPE)` para explorar combinaciones óptimas de hiperparámetros.
   - Registro de experimentos y métricas durante el proceso de optimización en **MLflow**.

---

#### **Comparación y Selección de Modelos (*Champion* y *Challenger*)**
- Cada modelo fue evaluado utilizando los datos de prueba, calculando métricas de desempeño como el **RMSE** para determinar su efectividad.
- Los modelos fueron comparados en términos de precisión y eficiencia computacional.
- Se seleccionó el modelo con el mejor desempeño como el **Champion**, etiquetándolo como el modelo principal en producción.
- Modelos alternativos con buen desempeño fueron registrados como **Challengers**, preparados para futuras pruebas comparativas.

---

#### **Registro en el Model Registry**
- El modelo **Champion** fue registrado en el **Model Registry** de MLflow.
- Se asignó la etapa `Production` al modelo Champion, garantizando su uso en aplicaciones y despliegues futuros.
- Los modelos Challengers fueron almacenados en el registro con el alias `Staging`, listos para ser promovidos a producción si demuestran un mejor desempeño en nuevas evaluaciones.
- Toda la información, incluidos hiperparámetros, métricas y artefactos, fue documentada en **MLflow**, asegurando trazabilidad y reproducibilidad.

Este proceso permite mantener un control estricto sobre el desempeño de los modelos, asegurando que el sistema siempre utilice la versión más eficiente y precisa para las predicciones.

---

### 6. Pipelines de Automatización y Tracking

#### **Orquestación con Prefect**
El proyecto utiliza **Prefect**, una plataforma para la orquestación de flujos de trabajo, con el objetivo de automatizar las tareas clave en el pipeline de ciencia de datos. Las principales tareas gestionadas incluyen:

1. **Lectura y Preprocesamiento de Datos:**
   - Los datos son cargados desde el directorio `data/raw` y divididos en conjuntos de entrenamiento y prueba.
   - Se aplica estandarización a los datos mediante `StandardScaler` para garantizar un mejor rendimiento de los modelos.

2. **Optimización de Hiperparámetros:**
   - Se realiza una búsqueda de hiperparámetros utilizando el método `Tree-structured Parzen Estimator (TPE)` de `hyperopt`.
   - Los parámetros óptimos son registrados en **MLflow** para futuras referencias.

3. **Entrenamiento del Modelo:**
   - Se entrena el modelo con los mejores hiperparámetros identificados.
   - Las métricas de desempeño, como el **Root Mean Squared Error (RMSE)**, son calculadas y registradas.

4. **Selección del Mejor Modelo:**
   - Se evalúan los modelos en el experimento correspondiente de **MLflow**.
   - El mejor modelo es registrado en el **Model Registry** como **Challenger**.

5. **Actualización del Modelo Champion:**
   - Si el modelo Challenger demuestra ser superior al Champion actual, se promueve a producción, garantizando que la mejor versión del modelo esté siempre activa.

#### **Ejecución Local y Configuración con MLflow**
El proyecto está configurado para integrar **MLflow** tanto para el seguimiento de experimentos como para la gestión de modelos. Los pasos clave incluyen:

1. **Configuración de MLflow:**
   - El seguimiento de experimentos se gestiona localmente y también en **Dagshub**, utilizando su URI de rastreo.
   - Los artefactos, métricas y parámetros de cada ejecución se registran automáticamente.

2. **Gestión de Modelos:**
   - Los modelos entrenados se registran con sus métricas asociadas.
   - El mejor modelo es etiquetado como **Champion** en el **Model Registry** de MLflow, mientras que las versiones alternativas son almacenadas como **Challengers**.

3. **Visualización de Experimentos:**
   - Los resultados pueden ser visualizados en la interfaz de MLflow mediante el comando `mlflow ui`, o en **Dagshub** para colaborar con otros usuarios del proyecto.

#### **Beneficios del Pipeline Automatizado**
- **Consistencia:** Asegura que los pasos de preprocesamiento, entrenamiento y evaluación se ejecuten de manera uniforme.
- **Reproducibilidad:** Los experimentos, parámetros y métricas están completamente registrados.
- **Escalabilidad:** Puede extenderse fácilmente para manejar nuevos datos o modelos.
- **Colaboración:** La integración con **Dagshub** permite a los equipos trabajar de manera conjunta, visualizando y gestionando los experimentos en un entorno centralizado.

Este pipeline combina la potencia de Prefect para la orquestación y MLflow para el seguimiento, garantizando un flujo de trabajo eficiente y escalable.

---

### 7. Desarrollo de la Aplicación

#### **Interfaz de Usuario (Frontend)**
La interfaz de usuario fue desarrollada utilizando **Streamlit**, una herramienta de código abierto que permite construir aplicaciones web interactivas para proyectos de ciencia de datos de manera rápida y sencilla. La interfaz permite a los usuarios ingresar valores correspondientes a las variables del modelo (como `co`, `humidity`, `lpg`, `smoke`, etc.) y obtener una predicción de la temperatura en grados centígrados.

- **Características Principales:**
  - Diseño minimalista y fácil de usar.
  - Entrada de datos a través de un panel lateral (`st.sidebar`) que permite personalizar las variables.
  - Botón de predicción que envía una solicitud POST al backend para calcular la temperatura.
  - Visualización en tiempo real de los resultados directamente en la interfaz.

- **Funcionamiento:**
  1. Los usuarios ingresan los valores de las variables de entrada.
  2. La aplicación envía una solicitud al backend para obtener la predicción de la temperatura.
  3. La respuesta del backend (predicción) se muestra en la página principal.

- **Puerto Utilizado:**
  - La aplicación se ejecuta en el puerto `8501`.

---

#### **Modelo y API (Backend)**
El backend fue implementado utilizando **FastAPI**, un framework moderno para construir APIs rápidas y robustas en Python. Este componente gestiona las solicitudes de predicción enviadas desde el frontend y ejecuta el modelo registrado en **MLflow** para devolver el resultado.

- **Características Principales:**
  - Carga dinámica del modelo etiquetado como `champion` desde el **Model Registry** de MLflow.
  - API RESTful con un endpoint principal `/predict` que recibe los datos de entrada, realiza las predicciones y devuelve el resultado.
  - Validación automática de los datos de entrada utilizando clases de datos definidas con `pydantic`.

- **Funcionamiento:**
  1. El modelo `champion` es cargado dinámicamente desde MLflow al iniciar el servicio.
  2. Cuando se recibe una solicitud en el endpoint `/predict`, los datos se convierten en un DataFrame y se pasan al modelo para generar la predicción.
  3. El resultado es devuelto al frontend en formato JSON.

- **Puerto Utilizado:**
  - El backend se ejecuta en el puerto `8000`.

---

#### **Despliegue con Docker**
Ambos servicios, frontend y backend, están empaquetados en contenedores Docker para garantizar portabilidad y facilidad de despliegue.

- **Archivo `docker-compose.yml`:**
  - Define los servicios de frontend (`ui`) y backend (`model`).
  - Especifica las carpetas donde se encuentran los archivos necesarios para construir las imágenes Docker (`context`).
  - Configura los puertos para acceder a cada servicio:
    - Frontend: `8501`
    - Backend: `8000`
  - Garantiza que el servicio del backend esté disponible antes de iniciar el frontend utilizando la directiva `depends_on`.

- **Ventajas del Uso de Docker:**
  - Simplifica el proceso de implementación en diferentes entornos.
  - Asegura que todos los componentes se ejecuten con las dependencias correctas.
  - Facilita la escalabilidad, permitiendo desplegar múltiples instancias en caso de necesidad.

Este diseño modular y contenedorizado garantiza que la aplicación sea fácil de mantener, implementar y escalar, permitiendo a los usuarios interactuar con el modelo de predicción de temperatura de manera rápida y eficiente.

---

### 8. Conclusiones y Reflexiones Finales

Este proyecto representó una oportunidad invaluable para aplicar de manera práctica los conocimientos adquiridos en la clase **Proyecto de Ciencia de Datos**. Desde el análisis y procesamiento de datos hasta la implementación de modelos predictivos y el desarrollo de una aplicación completa, se abordaron múltiples aspectos que abarcan todo el ciclo de vida de un proyecto de ciencia de datos.

#### **Conclusiones**
1. **Dominio del Flujo de Trabajo en Ciencia de Datos:**  
   El proyecto permitió comprender la importancia de estructurar los datos y flujos de trabajo en diferentes etapas, desde los datos crudos (`raw`) hasta los modelos en producción. Esta organización garantiza la reproducibilidad y el mantenimiento del proyecto a largo plazo.

2. **Automatización y Seguimiento:**  
   La integración de herramientas como **Prefect** y **MLflow** facilitó la automatización y el seguimiento del pipeline, asegurando una ejecución eficiente y trazabilidad en cada experimento realizado. La capacidad de gestionar versiones de modelos con MLflow demostró ser crucial para mantener un sistema robusto.

3. **Precisión y Optimización:**  
   La experimentación con diferentes modelos y la optimización de hiperparámetros llevaron a obtener predicciones precisas para la temperatura. La comparación entre modelos permitió seleccionar el mejor (Champion) y mantener opciones alternativas (Challenger) disponibles para futuras mejoras.

4. **Aplicación Interactiva:**  
   El desarrollo de una aplicación completa con **Streamlit** y **FastAPI** proporcionó un entorno intuitivo para los usuarios finales, demostrando cómo los modelos de machine learning pueden integrarse en soluciones prácticas y accesibles.

#### **Reflexiones Finales**
1. **Impacto del Proyecto:**  
   Este trabajo subraya cómo la ciencia de datos puede aprovechar los datos generados por dispositivos IoT para resolver problemas reales, como el monitoreo ambiental. Aplicaciones como esta tienen un potencial significativo en sectores como la agricultura, la salud y la gestión de recursos.

2. **Retos Superados:**  
   Durante el desarrollo, se enfrentaron desafíos técnicos como la limpieza de datos, la gestión de clases desbalanceadas y la implementación de un pipeline automatizado. Superar estos retos fortaleció la comprensión técnica y práctica del ciclo de vida de un proyecto de ciencia de datos.

3. **Aprendizajes Clave:**  
   - La importancia de documentar cada paso del proyecto para garantizar su reproducibilidad.
   - El valor de las herramientas de seguimiento como MLflow y Dagshub para la colaboración en equipo.
   - La necesidad de evaluar y comparar continuamente modelos para garantizar el mejor rendimiento.

4. **Perspectivas Futuras:**  
   Este proyecto puede ser ampliado con:
   - Más datos para mejorar la generalización de los modelos.
   - Integración con servicios en la nube para procesar datos en tiempo real.
   - Mejores visualizaciones en el frontend para enriquecer la experiencia del usuario.

#### **Conclusión Final**
Este proyecto consolidó los conocimientos adquiridos a lo largo del curso y demostró cómo un enfoque bien estructurado en ciencia de datos puede transformar datos complejos en soluciones prácticas. Más allá de ser un logro académico, representa un paso hacia el desarrollo de aplicaciones basadas en inteligencia artificial que pueden generar un impacto real en diversos sectores.

